In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "t001": MTU.get_fully_qualified_table_name("oracle", "crd", "vw_t001_base_table"),
    "cmpy_rpt_grp": MTU.get_fully_qualified_table_name("external", "static", "cmpy_rpt_grp"),
}

In [0]:
# mapping from silver tables to gold company table
cmpy_col_mappings = {
    "cmpy_cd": F.col("t001.company_code"),
    "sap_cmpy_cd": F.col("t001.rcomp"),
    "cmpy_nm": F.col("t001.company_code_desc"),
    "city_nm": F.col("t001.city"),
    "cntry_cd": F.col("t001.land1"),
    "crncy_cd": F.col("t001.waers"),
    "chart_of_accts_cd": F.col("t001.ktopl"),
    "fiscl_yr_varnt_cd": F.col("t001.periv"),
    "regn_cd": F.col("t001.region"),
    "sort_1_cd": F.col("t001.sort1"),
    "sort_2_cd": F.col("t001.sort2"),
    "tax_jrsdctn_cd": F.col("t001.taxjurcode"),
    "dst_nm": F.col("t001.district"),
    "cmpy_long_nm": F.col("t001.name"),
    "st_addr_txt": F.col("t001.street"),
    "cmpy_grpg_cd": F.col("cmpy_rpt_grp.cmpy_grp_cd"),
}

In [0]:
try:
    # Base DF for join
    left_df = spark.read.table(SOURCE_TABLES["t001"]).alias("t001")
    # other table to join to base
    cmpy_rpt_grp_df = (
        spark.read.table(SOURCE_TABLES["cmpy_rpt_grp"]).select("cmpy_cd", "cmpy_grp_cd").alias("cmpy_rpt_grp")
    )

    # join silver table
    left_df = left_df.join(
        other=cmpy_rpt_grp_df, on=F.col("t001.company_code") == F.col("cmpy_rpt_grp.cmpy_cd"), how="left"
    ).drop("cmpy_rpt_grp.cmpy_cd")

    # Transform and select final columns
    medallion.df = (
        left_df.withColumns(cmpy_col_mappings)
        .select(*cmpy_col_mappings.keys())
        .withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
    )

except Exception as e:
    medallion.logger.error(f"Error processing company source table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing company table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )